**Universidade Federal do Pampa (Unipampa) — Campus Alegrete**  
AL5170 — Introdução à Ciência de Dados  
Prof. João Pablo S. da Silva

---

# Notebook 1 — Limpeza de Dados

**Dataset:** Food Wastage Data in Restaurant  
**Fonte:** Kaggle — https://www.kaggle.com/datasets/trevinhannibal/food-wastage-data-in-restaurant  
**Grupo 3:** Arthur Provenzi Parizotto · Bianca Maia Ribeiro · Emilly Nascimento Dias · Rafael Jacques do Nascimento Lopes  
**Projeto:** Investigação do Desperdício de Alimentos em Serviços de Buffet

---

Este notebook documenta todo o processo de diagnóstico e limpeza do dataset, produzindo a **versão silver** a partir da **versão bronze**. Cada decisão é justificada tecnicamente, garantindo rastreabilidade e reprodutibilidade do tratamento.


## 1. Importação de Pacotes

Importa os pacotes necessários para manipulação e diagnóstico dos dados:
- `pandas`: manipulação de DataFrames tabulares.
- `numpy`: suporte a operações numéricas.
- `warnings`: supressão de alertas não críticos.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 2. Carregamento da Fonte e Geração da Camada Bronze


A fonte do conjunto de dados é o Kaggle, no dataset **Food Wastage Data in Restaurant**, disponibilizado em:

https://www.kaggle.com/datasets/trevinhannibal/food-wastage-data-in-restaurant

Para executar este notebook, o arquivo original baixado do Kaggle deve estar na mesma pasta do notebook com o nome **`food_wastage_data.csv`**. O notebook lê apenas esse arquivo de entrada e, em seguida, materializa a camada **bronze** em `food_wastage_bronze.csv`.

A camada bronze representa a cópia rastreável do dado original, sem padronização de nomes, sem conversão de tipos e sem remoção de registros. Segundo a descrição do Kaggle, o dataset reúne dados de desperdício de alimentos de um restaurante localizado na Índia, organizado para fins de pesquisa sobre padrões e magnitude do desperdício. Como limitação inicial, a fonte não informa identificador único dos eventos, unidade explícita das quantidades, data de coleta, instrumento de coleta ou garantia de representatividade estatística.


In [ ]:
INPUT_FILE = 'food_wastage_data.csv'
BRONZE_FILE = 'food_wastage_bronze.csv'

df_bronze = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')

# Materialização da camada bronze: cópia do arquivo original para rastreabilidade do pipeline.
df_bronze.to_csv(BRONZE_FILE, index=False)

print(f"Arquivo de entrada lido: {INPUT_FILE}")
print(f"Dataset bronze gerado: {BRONZE_FILE}")
print(f"Dimensões do dataset bronze: {df_bronze.shape[0]} linhas × {df_bronze.shape[1]} colunas")
df_bronze.head()


## 3. Diagnóstico de Qualidade dos Dados

O diagnóstico mapeia os problemas existentes no dataset antes de qualquer intervenção. Ele é o fundamento técnico que justifica cada decisão de limpeza posterior. São avaliados: estrutura geral, tipos de dados, valores ausentes, duplicatas, categorias e consistência lógica.

### 3.1 Estrutura Geral e Tipos de Dados

Inspeciona os tipos de dados de cada coluna. Tipos incorretos podem causar erros silenciosos em operações numéricas e agrupamentos, por isso devem ser identificados antes de qualquer transformação.

In [ ]:
print("=== Informações Gerais ===")
df_bronze.info()

### 3.2 Estatísticas Descritivas

Gera um resumo estatístico das variáveis numéricas, incluindo o percentil 99, útil para identificar valores extremos (outliers). Variáveis categóricas são inspecionadas quanto à quantidade de categorias únicas.

In [ ]:
print("=== Estatísticas Descritivas — Variáveis Numéricas ===")
display(df_bronze.describe(percentiles=[0.25, 0.50, 0.75, 0.99]))

print("\n=== Variáveis Categóricas — Contagem de Categorias Únicas ===")
cat_cols = df_bronze.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f"  {col}: {df_bronze[col].nunique()} categorias → {sorted(df_bronze[col].unique())}")

### 3.3 Diagnóstico de Valores Ausentes

Define e executa a função `compute_missing_data`, que calcula o total e a proporção de valores ausentes por coluna. A ausência de dados pode comprometer indicadores como taxa de desperdício, exigindo tratamento antes das análises.

In [ ]:
def compute_missing_data(df):
    na_count = df.isna().sum()
    na_rate = (df.isna().mean() * 100).round(2)
    report = (
        pd.DataFrame({'missing': na_count, 'missing_%': na_rate})
        .sort_values('missing_%', ascending=False)
    )
    return report

print("=== Relatório de Valores Ausentes ===")
print(compute_missing_data(df_bronze))

### 3.4 Diagnóstico de Registros Duplicados

Verifica a existência de linhas completamente idênticas em todas as colunas. Como o dataset não possui um identificador único por evento, a duplicidade considerada nesta preparação é **duplicidade completa**: registros com os mesmos valores nos 11 atributos originais.

Esse critério é conservador porque não remove registros parecidos, apenas registros exatamente iguais. Ainda assim, há uma limitação: sem uma chave primária do evento, não é possível provar com certeza absoluta se uma linha idêntica é erro de registro ou repetição real de um evento com os mesmos atributos.


In [ ]:
n_dup = df_bronze.duplicated().sum()
print(f"Registros duplicados encontrados: {n_dup} ({n_dup/len(df_bronze)*100:.1f}% do total)")
print(f"Linhas únicas: {df_bronze.drop_duplicates().shape[0]}")

print("\n=== Exemplo de registros duplicados ===")
df_bronze[df_bronze.duplicated(keep=False)].sort_values(list(df_bronze.columns)).head(8)

### 3.5 Diagnóstico de Inconsistências Lógicas

Verifica se `Wastage Food Amount` (desperdício) supera `Quantity of Food` (quantidade produzida), o que seria logicamente impossível — não é possível desperdiçar mais do que foi produzido. Também verifica valores negativos ou nulos nas variáveis numéricas.

In [ ]:
# Desperdício maior que a quantidade produzida
wastage_gt_qty = (df_bronze['Wastage Food Amount'] > df_bronze['Quantity of Food']).sum()
print(f"Registros com Wastage > Quantity of Food: {wastage_gt_qty}")

# Valores nulos ou negativos nas variáveis numéricas
num_cols = ['Number of Guests', 'Quantity of Food', 'Wastage Food Amount']
for col in num_cols:
    neg = (df_bronze[col] <= 0).sum()
    print(f"Valores <= 0 em '{col}': {neg}")

## 4. Limpeza dos Dados

Com base no diagnóstico, são executadas as decisões de limpeza, cada uma com justificativa técnica e registro do impacto sobre o dataset.

### 4.1 Cópia de Trabalho

O DataFrame bronze é copiado para `df_silver`. Esta separação garante rastreabilidade: o bronze permanece intacto como ponto de entrada, e todas as alterações são feitas sobre a cópia, permitindo comparar antes e depois em qualquer etapa.

In [ ]:
df_silver = df_bronze.copy()
print(f"Cópia criada. Dimensões iniciais: {df_silver.shape}")

### 4.2 Padronização dos Nomes de Colunas

Os nomes originais das colunas contêm espaços e letras maiúsculas, o que dificulta o acesso programático no pandas. A padronização converte para `snake_case` (letras minúsculas, espaços substituídos por underscores), tornando o código mais legível e evitando erros por nomes inconsistentes.

In [ ]:
def to_snake_case(name):
    return name.strip().lower().replace(' ', '_')

original_cols = df_silver.columns.tolist()
df_silver.columns = [to_snake_case(c) for c in df_silver.columns]

print("=== Mapeamento de renomeação ===")
for old, new in zip(original_cols, df_silver.columns):
    print(f"  '{old}' → '{new}'")

### 4.3 Correção de Tipos de Dados

As colunas numéricas já estão como `int64`. As colunas categóricas são convertidas para o tipo `category` do pandas, que reduz o consumo de memória e torna agrupamentos mais eficientes. Essa conversão não altera os valores, apenas a representação interna.

In [ ]:
cat_cols = ['type_of_food', 'event_type', 'storage_conditions',
            'purchase_history', 'seasonality', 'preparation_method',
            'geographical_location', 'pricing']

for col in cat_cols:
    df_silver[col] = df_silver[col].astype('category')

print("=== Tipos após conversão ===")
print(df_silver.dtypes)

### 4.4 Tratamento de Valores Ausentes

O diagnóstico confirmou ausência de valores nulos no dataset. Por completude técnica, registra-se aqui que nenhuma intervenção foi necessária. Essa verificação pós-padronização garante que a conversão de tipos não introduziu NaN onde não existia.

In [ ]:
print("=== Verificação de ausentes após padronização ===")
print(compute_missing_data(df_silver))
print("\nNenhum valor ausente identificado. Nenhuma intervenção necessária.")

### 4.5 Remoção de Registros Duplicados

**Critério de remoção aplicado:** foram removidas somente duplicatas completas, isto é, linhas com valores idênticos em todas as 11 colunas originais. A primeira ocorrência foi preservada (`keep='first'`) e as repetições posteriores foram descartadas.

**Regra que causou perda de registros:** nenhuma linha foi removida por ausência de dados, valores negativos ou inconsistência lógica. A redução do dataset de 1.782 para 1.618 registros ocorre exclusivamente pela remoção de 164 duplicatas completas.

**Impacto da decisão:** a remoção evita que eventos repetidos inflem contagens, médias e taxas de desperdício nas análises posteriores. Como efeito colateral, caso alguma repetição completa representasse um evento real distinto, esse evento deixaria de estar representado na camada silver. Essa possibilidade permanece registrada como limitação por causa da ausência de identificador único.


In [ ]:
n_before = df_silver.shape[0]
n_missing_before = df_silver.isna().sum().sum()
n_logic_invalid_before = (df_silver['wastage_food_amount'] > df_silver['quantity_of_food']).sum()

# Remoção exclusiva de duplicatas completas.
df_silver = df_silver.drop_duplicates(keep='first').reset_index(drop=True)
n_after = df_silver.shape[0]
n_removed = n_before - n_after

impacto_remocao = pd.DataFrame({
    'Critério avaliado': [
        'Valores ausentes',
        'Desperdício maior que quantidade produzida',
        'Duplicatas completas removidas'
    ],
    'Registros removidos': [
        0,
        0,
        n_removed
    ],
    'Observação': [
        f'{n_missing_before} valores ausentes encontrados; nenhuma linha removida por este critério.',
        f'{n_logic_invalid_before} inconsistências encontradas; nenhuma linha removida por este critério.',
        'Mantida a primeira ocorrência de cada registro completo duplicado.'
    ]
})

print(f"Registros antes: {n_before}")
print(f"Registros removidos: {n_removed} ({n_removed/n_before*100:.1f}%)")
print(f"Registros após limpeza: {n_after}")
print("\n=== Impacto dos critérios de limpeza ===")
print(impacto_remocao.to_string(index=False))


### 4.6 Verificação de Inconsistências Lógicas

Reitera a verificação de inconsistências após remoção de duplicatas. O diagnóstico original não encontrou registros com desperdício superior à quantidade produzida. A ausência de registros inválidos confirma que não há necessidade de filtragem adicional por inconsistência lógica.

In [ ]:
# Verificação pós-limpeza
wastage_gt_qty = (df_silver['wastage_food_amount'] > df_silver['quantity_of_food']).sum()
print(f"Registros com wastage > quantity_of_food: {wastage_gt_qty}")

for col in ['number_of_guests', 'quantity_of_food', 'wastage_food_amount']:
    neg = (df_silver[col] <= 0).sum()
    print(f"Valores <= 0 em '{col}': {neg}")

print("\nNenhuma inconsistência lógica identificada. Dataset consistente.")

## 5. Geração do Dataset Silver

O dataset silver representa a base limpa e padronizada para as próximas etapas do pipeline. Ele mantém a mesma unidade de análise da bronze — um registro/evento de desperdício — mas com nomes de colunas padronizados, tipos categóricos ajustados e duplicatas completas removidas. É salvo em formato CSV para portabilidade e rastreabilidade.


In [ ]:
df_silver.to_csv('food_wastage_silver.csv', index=False)
print(f"Dataset silver gerado com sucesso.")
print(f"Dimensões finais: {df_silver.shape[0]} linhas × {df_silver.shape[1]} colunas")
df_silver.head()

## 6. Comparação Bronze × Silver

Confronta as versões bronze e silver para evidenciar o impacto das intervenções realizadas. A comparação serve como evidência técnica das decisões tomadas.

In [ ]:
comparison = pd.DataFrame({
    'Aspecto': ['Linhas', 'Colunas', 'Duplicatas', 'Valores ausentes', 'Nomes das colunas', 'Tipos categóricos'],
    'Bronze': [
        df_bronze.shape[0],
        df_bronze.shape[1],
        df_bronze.duplicated().sum(),
        df_bronze.isna().sum().sum(),
        'Espaços, CamelCase',
        'object (str)'
    ],
    'Silver': [
        df_silver.shape[0],
        df_silver.shape[1],
        df_silver.duplicated().sum(),
        df_silver.isna().sum().sum(),
        'snake_case',
        'category'
    ]
})
print(comparison.to_string(index=False))

## 7. Limitações da Preparação Realizada

**1. Fonte secundária e contexto de coleta limitado:** o dataset foi obtido no Kaggle e não traz documentação completa sobre instrumento de coleta, período, responsável pela coleta, controle de qualidade original ou critérios de amostragem.

**2. Ausência de identificador único por evento:** o dataset não possui chave primária explícita. Por isso, a remoção de duplicatas completas é tecnicamente justificável, mas não elimina totalmente a possibilidade de dois eventos reais terem sido registrados com os mesmos valores.

**3. Unidade das quantidades não explicitada:** as colunas `quantity_of_food` e `wastage_food_amount` são tratadas como medidas na mesma unidade, mas a fonte não declara se os valores representam kg, porções ou outra unidade operacional.

**4. Ausência de variável temporal:** o dataset não contém data dos eventos. Assim, não é possível verificar tendência histórica, variação por mês, efeito de feriados ou sazonalidade temporal real.

**5. Representatividade não comprovada:** a fonte informa o contexto de restaurante, mas não permite afirmar que os padrões observados sejam representativos de todos os restaurantes, buffets ou regiões.

**6. Limpeza não altera conteúdo semântico:** a camada silver melhora padronização e remove duplicidades completas, mas problemas estruturais da fonte, como falta de unidade, falta de data e falta de identificador único, permanecem.
